# 02. Stream Processing - Kafka & Flink

Architecture: **producers → Apache Kafka → Flink jobs → sinks (e.g., Delta / dashboards)**. This notebook demonstrates Kafka I/O and windowing concepts; Flink is often run as a separate cluster job—here we **simulate** windowed analytics with pandas for clarity in a notebook.


In [ ]:
from kafka import KafkaProducer, KafkaConsumer
from IPython.display import display
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta


## Kafka Setup


In [ ]:
BOOTSTRAP = "localhost:9092"
TOPIC = "nyc-taxi-rides"

producer = KafkaProducer(
    bootstrap_servers=[BOOTSTRAP],
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
)

sample_events = [
    {
        "ride_id": "r1",
        "pickup_ts": time.time(),
        "fare": 12.5,
        "distance": 3.2,
        "zone": 161,
    },
    {"ride_id": "r2", "pickup_ts": time.time(), "fare": 8.0, "distance": 1.1, "zone": 48},
]
for ev in sample_events:
    producer.send(TOPIC, value=ev)
producer.flush()
print("Sent", len(sample_events), "events to", TOPIC)


In [ ]:
consumer = KafkaConsumer(
    TOPIC,
    bootstrap_servers=[BOOTSTRAP],
    auto_offset_reset="earliest",
    enable_auto_commit=True,
    group_id="aide2-demo-group",
    value_deserializer=lambda m: json.loads(m.decode("utf-8")),
    consumer_timeout_ms=5000,
)

msgs = []
for rec in consumer:
    msgs.append(rec.value)
    if len(msgs) >= 5:
        break
consumer.close()
pd.DataFrame(msgs)


## Stream Processing with Windowing


In [ ]:
# Window types (conceptual; Flink SQL would express these in a job):
# - Tumbling 5 min: fixed buckets, no overlap (e.g. [0-5), [5-10), ...).
# - Sliding 15 min / 5 min slide: each 15-minute window moves every 5 minutes (overlap).
# - Session 30 min gap: events grouped until 30 min of inactivity closes the session.

print("Tumbling / sliding / session windows are configured in Flink (EventTime + watermarks).")


In [ ]:
rng = pd.date_range("2024-01-01 08:00", periods=40, freq="1min")
rides = pd.DataFrame(
    {
        "event_ts": rng,
        "fare": 10 + pd.Series(range(40)) * 0.1 + np.random.default_rng(42).normal(0, 0.5, 40),
    }
)
rides["fare"] = rides["fare"].clip(lower=0)


In [ ]:
# Tumbling 5-minute buckets
rides = rides.set_index("event_ts")
tumbling = rides["fare"].resample("5min").agg(["count", "mean", "sum"])
display(tumbling.head(10))

fig, ax = plt.subplots(figsize=(8, 3))
tumbling["count"].plot(kind="bar", ax=ax, title="Tumbling 5-min trip counts (simulated)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# Sliding-style: rolling mean over 15 minutes, evaluated every row (approximates sliding window)
rides_reset = rides.reset_index()
rides_reset["ma_15m"] = rides_reset.set_index("event_ts")["fare"].rolling("15min").mean().values

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(rides_reset["event_ts"], rides_reset["fare"], label="fare", alpha=0.6)
ax.plot(rides_reset["event_ts"], rides_reset["ma_15m"], label="15m rolling mean", linewidth=2)
ax.legend()
plt.title("Sliding-window style moving average")
plt.tight_layout()
plt.show()


## Anomaly Detection


In [ ]:
fare_series = rides_reset["fare"]
mu, sigma = fare_series.mean(), fare_series.std(ddof=0)
z = (fare_series - mu) / (sigma if sigma else 1.0)
threshold = 2.5
rides_reset["is_anomaly"] = z.abs() > threshold


In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))
normal = rides_reset[~rides_reset["is_anomaly"]]
anom = rides_reset[rides_reset["is_anomaly"]]
ax.plot(normal["event_ts"], normal["fare"], label="normal", color="C0")
ax.scatter(anom["event_ts"], anom["fare"], color="red", label="flagged", zorder=5)
ax.axhline(mu, color="gray", linestyle="--", alpha=0.7)
ax.legend()
plt.title("Fare time series: normal vs Z-score anomalies")
plt.tight_layout()
plt.show()


In [ ]:
print("Anomaly rate:", rides_reset["is_anomaly"].mean())
print(rides_reset.loc[rides_reset["is_anomaly"], ["event_ts", "fare", "is_anomaly"]].describe(include="all"))


## Summary

Real-time pipelines combine **Kafka** for durable streaming, **Flink** (or Spark Structured Streaming) for stateful windowing and joins, and **lakehouse sinks** for training data. This notebook isolated **producer/consumer** patterns and **offline** window/anomaly analogs you can map to Flink SQL or the DataStream API.
